# 03 · 평가
**4세션 · 평가** — 실습 50분

고쳤다는 느낌은 증거가 아닙니다. **같은 문항, 같은 기준, 같은 방법으로** 다시 재야 고친 것입니다.

02에서 만든 최종 프롬프트 **`prompt_4`**를 정답이 있는 20문항에 돌려 점수를 내고(1차), 가장 많이 틀리는 유형 하나를 고쳐 **`prompt_5`**로 저장해 다시 측정하고(2차), 한 번 더 고쳐 **`prompt_6`**(2차 반복)으로 다시 측정합니다. 번호는 02에서 이어집니다.

| 순서 | 할 일 | 시간 |
|---|---|---|
| 1 | 20문항 — 구성표와 통과 조건을 읽는다 | 10분 |
| 2 | 1차 — `prompt_4` 채점, 실패 문항을 직접 연다 | 10분 |
| 3 | 2차 — 한 유형씩 고쳐 `prompt_5` · `prompt_6`, 같은 20문항으로 다시 채점 | 30분 |

**읽는 법** — 실습 셀마다 바로 앞에 **▶ 이 셀에서 할 것**, 바로 뒤에 **✔ 이 셀의 시사점**이 있습니다. 셀을 돌리기 전에 ▶를, 돌린 뒤에 출력과 ✔를 같이 읽습니다.

**▶ 이 셀에서 할 것** — 준비. 공용 도구 `llm`과, 채점 코드 `src/grade.py`에서 이 노트북이 쓰는 함수들을 불러옵니다.
- `load_tests` — 20문항(`data/tests.jsonl`)을 읽는다
- `check` — 모델 답 하나를 통과 조건 하나로 판정한다
- `run` — 프롬프트 하나로 20문항을 전부 돌려 채점한다
- `summarize` · `save` · `print_table` · `print_fails` — 결과를 요약하고, 결과표에 저장하고, 표와 실패 목록을 찍는다

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
import json
from pathlib import Path
from collections import Counter
from src.grade import load_tests, run, check, summarize, save, print_table, print_fails

**✔ 이 셀의 시사점**
- 출력의 모델 이름과 `실제 호출`을 확인합니다. 02와 같은 모델이어야 02의 결과와 이어서 비교할 수 있습니다.

## 1. 고정 테스트 20문항
`data/tests.jsonl` — 한 줄에 한 문항. **프롬프트를 고치기 전에 만들고, 고치는 동안 바꾸지 않습니다** (고치는 중간에 문항이 바뀌면 전후 비교가 안 됩니다).

20문항은 3종류로 나뉘어 있고, 종류마다 보려는 게 다릅니다.

| 구성 | 문항 수 | 이 실습의 문항 | 뭘 보려고 넣었나 |
|---|---|---|---|
| 평범한 입력 | 10 | 1–10 | 가장 흔한 문의에서 기본으로 되는지 |
| 경계에 있는 입력 | 7 | 11 구매 요청 · 12 차분한 어투의 위험 신호 · 13 장비 고장 · 14 격한 어투의 예약 시스템 문의 · 15 휴가 중 급여 · 19 영어로 쓴 급여 문의 · 20 사라진 회의실 예약 | 말투·단어·언어에 속지 않고 이 회사 기준대로 판단하는지 |
| 어느 카테고리에도 안 맞는 입력 | 3 | 16 · 17 · 18 | 억지로 짜맞추지 않고 "기타"로 분류하는지 |

여기 적힌 **정답(`check`)은 설계자가 정한 판단 기준**이지 객관적 사실이 아닙니다(02의 4절). 모델이 다르게 답했을 때 프롬프트를 고칠지 정답을 고칠지는 근거를 보고 사람이 정합니다.

| check 필드 | 뜻 |
|---|---|
| `category` | 이 값과 같은 카테고리여야 통과 |
| `urgency` | 이 값과 같은 긴급도여야 통과 |
| `has` | 요약 어딘가에 이 말이 있어야 통과 (없으면 생략) |

**▶ 이 셀에서 할 것** — 20문항을 읽어 종류별 문항 수를 세고, 종류마다 첫 문항 하나씩 **문의 본문과 통과 조건(`check`)**을 찍습니다. 모델은 부르지 않습니다.

In [ ]:
tests = load_tests()
print("종류별 문항 수:", dict(Counter(t["group"] for t in tests)), "\n")

# 종류마다 첫 문항 하나씩 — 문의 본문과 통과 조건(check)이 어떻게 적혀 있는지 본다
for group in ["평범", "경계", "기타"]:
    t = next(t for t in tests if t["group"] == group)
    print(f"[{group}] #{t['id']:>2}  {t['input']}")
    print(f"        check = {t['check']}\n")

**✔ 이 셀의 시사점**
- 종류별 문항 수가 위 표대로 10 · 7 · 3인지 확인합니다.
- `check`에 적힌 `category` · `urgency` (· `has`)가 곧 **채점 기준**입니다. 위 표의 설명이 실제 파일에서는 이런 모양으로 적혀 있습니다.

## 2. check — 판정 규칙
`check(출력, 통과 조건)` → `(통과 여부, 실패 유형, 설명)`. 판정 순서: **형식 위반 → 지어냄 → 지시 일부 누락 → 사실 오류** — 앞에서 걸리면 거기서 멈춥니다.

**▶ 이 셀에서 할 것** — 모델 없이, 일부러 틀리게 만든 **가짜 답 6개**를 `check()`에 넣습니다. 가짜 답마다 "나와야 할 판정"과 "실제 판정"을 나란히 찍고, 둘이 같으면 ✓를 붙입니다.

In [ ]:
# 채점기를 먼저 시험한다 — 모델 없이, 일부러 틀리게 만든 가짜 답으로.
# (라벨, 어느 문항의 정답 조건으로 채점할지, 가짜 답, 나와야 할 판정)
spec1  = tests[0]["check"]     # 1번:  {"category": "IT", "urgency": "보통", "has": "비밀번호"}
spec16 = tests[15]["check"]    # 16번: {"category": "기타", "urgency": "낮음"}  ← 정답이 "기타"

fakes = [
    ("정상",            spec1,  '{"카테고리": "IT", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}',               None),
    ("코드 블록으로 감쌈", spec1,  '```json\n{"카테고리": "IT", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}\n```', "형식 위반"),
    ("보기에 없는 카테고리", spec1, '{"카테고리": "기술지원", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}',           "형식 위반"),
    ("요약에 필수 단어 없음", spec1, '{"카테고리": "IT", "긴급도": "보통", "요약": "로그인 문제 접수"}',                  "지시 일부 누락"),
    ("카테고리가 정답과 다름", spec1, '{"카테고리": "비품", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}',             "사실 오류"),
    ("기타여야 하는데 지어냄", spec16, '{"카테고리": "인사", "긴급도": "낮음", "요약": "워크숍 일정 문의"}',               "지어냄"),
]

print(f"{'가짜 답':<14}{'기대 판정':<10}{'실제 판정':<10} 설명")
print("-" * 70)
for label, spec, text, expected in fakes:
    ok, kind, why = check(text, spec)
    mark = "✓" if kind == expected else "✗"
    print(f"{label:<14}{str(expected or '통과'):<10}{str(kind or '통과'):<10} {mark}  {why}")

**✔ 이 셀의 시사점**
- 여섯 줄이 모두 ✓면, 채점기가 네 유형을 의도대로 가른다는 뜻입니다.
- 모델을 채점하기 **전에** 채점기부터 시험하는 이유: 채점기가 틀리면 그 뒤의 점수가 전부 틀립니다.
- "설명" 칸을 봅니다. 실제 채점에서 실패 문항마다 이런 설명이 붙어, 왜 실패로 판정했는지 알 수 있습니다.

## 3. 1차 · `prompt_4` 채점
02에서 만든 최종 프롬프트 `prompts/prompt_4.txt`를 20문항에 돌립니다. 02를 아직 안 돌려 파일이 없으면 아래 셀이 **풀이본으로 대신 돌리고 그렇다고 알려 줍니다** — 흐름을 먼저 보고, 02를 돌린 뒤 다시 오면 됩니다.

**▶ 이 셀에서 할 것** — `prompt_4`로 20문항을 한 번씩 돌려 채점하고, 결과를 `results/scoreboard.csv`에 한 줄 저장합니다. 이어서 통과 수 · 실패 유형별 개수 · 실패 문항 목록을 찍습니다.

In [ ]:
P4 = Path("prompts/prompt_4.txt")
if not P4.exists():
    print("※ prompts/prompt_4.txt 가 없습니다 (02 노트북을 아직 안 돌림). 풀이본 solutions/prompts/prompt_4.txt 로 대신 돌립니다.\n")
    P4 = Path("solutions/prompts/prompt_4.txt")

passed, fails, rows = run(str(P4), detail=True)
s1 = summarize(rows, llm.MODEL)
save(str(P4), llm.MODEL, rows, s1, note="1차 · prompt_4")
print(f"통과 {passed} / {len(rows)}")
print(Counter(kind for _, kind in fails))
print_fails(rows)

**✔ 이 셀의 시사점**
- 02에서 문의 하나로 3/3 통과하던 프롬프트도, 정답이 있는 20문항에서는 틀리는 문항이 나올 수 있습니다. 02의 "통과"는 형식과 보기만 확인한 것이고, **맞게 골랐는지**는 여기서 처음 잽니다.
- `Counter(...)` 줄에서 실패가 **어느 유형에 몰리는지** 봅니다. 4절에서 고칠 대상이 이 유형입니다.
- 실패 문항의 **정답**을 보면 이 회사만의 기준이 드러납니다. 강사가 돌렸을 때는 세 번 모두 14 / 20이었고, #2(토너 부족 → 낮음) · #8 · #15 · #19(급여 문제 → 높음)의 긴급도와 #14 · #20(예약 시스템 → 시설)의 카테고리가 틀렸습니다. 모델이 일반 상식으로 판단해서 틀린 것이라, 이 회사의 기준을 프롬프트에 **적어 줘야** 고쳐집니다.
- 실패 목록에서 실패가 어느 **종류**(평범 · 경계 · 기타)에 몰리는지도 봅니다.

### 실패 문항을 직접 연다
통과 수만 보면 **새로 생긴 실패**를 놓칩니다. 실패 문항의 출력을 열어 유형이 맞는지 사람이 확인합니다.
(사실 오류는 코드가 카테고리·긴급도 문자열만 비교해 판정합니다 — 애매한 경우엔 사람이 다시 봅니다.)

**▶ 이 셀에서 할 것** — 실패 문항 하나(기본값: 첫 번째 실패)의 판정 · 설명과 **모델 출력 원문**을 찍습니다. ✏️ `FAIL_ID`를 바꿔 다른 문항도 열어 봅니다.

In [ ]:
FAIL_ID = fails[0][0] if fails else 1      # ✏️ 보고 싶은 문항 번호
row = next(r for r in rows if r["id"] == FAIL_ID)
print(f"#{row['id']} [{row['group']}] {row['kind']} — {row['why']}\n")
print(row["text"][:1200])

**✔ 이 셀의 시사점**
- 판정 설명(예: "긴급도 기대 '낮음', 출력 '보통'")과 모델 출력 원문을 대조해, **판정이 맞는지 사람이 확인합니다.**
- 모델이 틀린 것인지, 정답(`check`)이 애매한 것인지도 여기서 가립니다. 정답이 애매하면 `tests.jsonl`을 고칠 일이지 프롬프트를 고칠 일이 아닙니다(02의 4절).

## 4. 2차 · 한 유형씩 고쳐 `prompt_5`
1. 위 실패 중 **가장 많은 유형 하나**를 고른다
2. `prompt_4.txt`를 복사해 `prompts/prompt_5.txt`로 만들고, **그 유형에 맞는 것 하나만** 고친다 (아래 표)
3. 같은 20문항으로 `prompt_4`와 `prompt_5`를 나란히 비교한다 — 통과 수만이 아니라 **어느 문항이 새로 통과하고 어느 문항이 새로 실패했는지**
4. 나빠졌으면 되돌린다

| 유형 | 고치는 곳 |
|---|---|
| 형식 위반 | 출력 형식 지시 · 예시 · `<json>` 태그로 감싸게 하기 |
| 지시 일부 누락 | 요약에 꼭 들어가야 할 말을 지시에 명시 |
| 지어냄 | "기타" 기준을 분명히 — 뚜렷이 안 맞으면 억지로 짜맞추지 말라고 지시 |
| 사실 오류 | 카테고리 · 긴급도 판단 기준을 더 구체적으로 — 02의 4절에서 본 **근거**가 어디를 고칠지 알려 준다 |

아래 셀은 기본값으로 **풀이본 `prompt_5`**(긴급도에 이 회사의 기준을 적음 — 급여 문제는 높음, 소모품 부족·사소한 불편은 낮음)를 돌립니다. 내 `prompt_5`를 만들었으면 경로와 NOTE를 바꾸세요.

**▶ 이 셀에서 할 것** — `prompt_5`로 같은 20문항을 다시 채점해 결과표에 한 줄 저장하고, `prompt_4`와 나란히 놓은 표를 찍습니다. 마지막 줄에 **새로 통과한 문항 번호와 새로 실패한 문항 번호**를 찍습니다.

In [ ]:
P5 = "solutions/prompts/prompt_5.txt"      # ✏️ 내 것: "prompts/prompt_5.txt"
NOTE = "긴급도: 이 회사 기준(급여는 높음 · 소모품 부족은 낮음)을 적음"   # ✏️ 무엇을 바꿨는지 한 줄 — 결과표에 남는다

p2, f2, rows2 = run(P5, detail=True)
s2 = summarize(rows2, llm.MODEL)
save(P5, llm.MODEL, rows2, s2, note=NOTE)
print_table([("prompt_4", llm.MODEL, s1), ("prompt_5", llm.MODEL, s2)])

before = {r["id"] for r in rows  if r["ok"]}
after  = {r["id"] for r in rows2 if r["ok"]}
print("\n새로 통과:", sorted(after - before), "  새로 실패:", sorted(before - after))

**✔ 이 셀의 시사점**
- 고치려던 유형(여기서는 긴급도의 사실 오류)이 줄었는지 표에서 봅니다. 강사가 돌렸을 때는 세 번 모두 18 / 20 — 긴급도 실패가 모두 사라지고 #14 · #20(카테고리)만 남았습니다.
- 통과 수만 보지 말고 **"새로 실패"**를 봅니다. 하나를 고치면 전에 맞던 문항이 깨질 수 있습니다. 새로 실패한 문항이 있으면 위 셀의 `FAIL_ID`로 열어 원인을 봅니다. (그 문항을 열려면 `row = next(r for r in rows2 ...)`처럼 `rows2`에서 찾습니다.)
- 통과 수 차이가 1~2개뿐이면 프롬프트 효과가 아니라 흔들림일 수도 있습니다(01 "같은 질문 다섯 번"). 같은 프롬프트를 한 번 더 돌려 보면 가릴 수 있습니다.

### 2차를 한 번 더 → `prompt_6`
`prompt_5`의 실패 중 가장 많은 유형 하나를 골라 `prompt_6`을 만듭니다. 풀이본 `prompt_6`은 **카테고리 기준(회의실·주차 예약 시스템, 출입카드처럼 총무팀이 관리하는 건물 시스템은 IT가 아니라 시설)**을 더했습니다. 모델은 "시스템"이라는 말만 보고 IT로 분류하지만, 이 회사에서는 총무팀 소관입니다. 02에서 쓴 출입카드 문의도 이 기준에 해당합니다.

**▶ 이 셀에서 할 것** — `prompt_6`으로 같은 20문항을 다시 채점해 결과표에 저장하고, `prompt_4` · `prompt_5` · `prompt_6` 세 줄을 나란히 찍은 뒤 남은 실패 문항을 찍습니다.

In [ ]:
P6 = "solutions/prompts/prompt_6.txt"      # ✏️ 내 것: "prompts/prompt_6.txt"
NOTE = "카테고리: 총무팀이 관리하는 건물 시스템(예약·출입카드)은 시설"

p3, f3, rows3 = run(P6, detail=True)
s3 = summarize(rows3, llm.MODEL)
save(P6, llm.MODEL, rows3, s3, note=NOTE)
print_table([("prompt_4", llm.MODEL, s1), ("prompt_5", llm.MODEL, s2), ("prompt_6", llm.MODEL, s3)])
print_fails(rows3)

**✔ 이 셀의 시사점**
- 세 줄을 비교해 **어느 수정이 어느 유형을 줄였는지** 봅니다. 한 번에 하나씩 고쳤기 때문에 이렇게 나눠 볼 수 있습니다.
- 표 아래에 실패 목록이 찍히지 않았다면 20문항을 모두 통과한 것입니다.
- **1건 비용** 칸도 봅니다. 기준을 덧붙이면 프롬프트가 길어져 입력 토큰이 늘고, 1건 비용이 오를 수 있습니다. 통과가 늘어난 만큼의 값어치가 있는지가 판단 거리입니다.

**발표 — 한 문장으로**
> "___ 유형을 줄이려고 ___를 바꿨더니 통과가 __ → __ 가 됐고, 대신 ___ 가 생겼다(또는 생기지 않았다)."

## 5. 결과표
`results/scoreboard.csv`에 실행할 때마다 한 줄씩 쌓입니다. 엑셀로 열어도 됩니다. "바꾼 것" 칸에 NOTE가 들어가므로, 나중에 봐도 **어느 줄이 어떤 수정의 결과인지** 알 수 있습니다.

**▶ 이 셀에서 할 것** — `results/scoreboard.csv`의 마지막 8줄을 찍습니다. 줄이 아직 적으면 맨 앞에 칸 이름 줄도 함께 보입니다.

In [ ]:
import csv
with open("results/scoreboard.csv", encoding="utf-8-sig") as f:
    for row in list(csv.reader(f))[-8:]:
        print(" | ".join(row[:12]))

**✔ 이 셀의 시사점**
- 방금 돌린 1차(`prompt_4`) · 2차(`prompt_5` · `prompt_6`)가 한 줄씩 쌓여 있습니다. 노트북을 다시 돌리면 그 줄이 또 쌓입니다.
- 맨 끝 "바꾼 것" 칸에 NOTE가 남아 있어, 숫자만 봐도 어떤 수정의 결과인지 알 수 있습니다. 이 표가 "고쳤다"는 주장의 증거입니다.

## 내일 가지고 올 것
- `prompt_6.txt` · `tests.jsonl` · `grade.py` · 결과표(`results/scoreboard.csv`)